This code will perform the following, load the dataset manually labeled, and then predict using a mlp model

# **Import Libraries**

In [5]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from pathlib import Path
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# **MLP**

In [6]:
train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

X_train = train_df.drop(columns=["target"]).values
y_train = train_df["target"].values

X_val = val_df.drop(columns=["target"]).values
y_val = val_df["target"].values

X_test = test_df.drop(columns=["target"]).values
y_test = test_df["target"].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (348, 30)
y_train shape: (348,)
X_val shape: (75, 30)
y_val shape: (75,)
X_test shape: (75, 30)
y_test shape: (75,)


In [7]:
import joblib

label_encoder = joblib.load(
    "data/label_encoder.joblib"
)

In [ ]:
import json

def build_and_train_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    num_classes=3,
    epochs=50,
    batch_size=16,
    learning_rate=1e-3
):
    """
    Builds and trains a simple MLP
    for contextual dirtiness classification.
    """

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(X_train.shape[1],)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            32,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax"
        )
    ])

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="sparse_categorical_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    model.summary()

    history = model.fit(

        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=epochs,
        batch_size=batch_size
    )

    return model, history


# ====================================================
# EVALUATION FUNCTION
# ====================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    label_encoder,
    model_name="mlp_model",
    results_dir="results"
):
    """
    Evaluates MLP model and saves:

    - metrics json
    - confusion matrix csv
    - trained model
    """

    # ------------------------------------------------
    # CREATE RESULTS DIRECTORY
    # ------------------------------------------------

    results_dir = Path(results_dir)

    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------
    # INFERENCE
    # ------------------------------------------------

    start_time = time.time()

    y_probs = model.predict(X_test)

    end_time = time.time()

    inference_time = (
        end_time - start_time
    )

    # ------------------------------------------------
    # PREDICTIONS
    # ------------------------------------------------

    y_pred = np.argmax(
        y_probs,
        axis=1
    )

    # ------------------------------------------------
    # METRICS
    # ------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    # ------------------------------------------------
    # METRICS DICTIONARY
    # ------------------------------------------------

    metrics = {

        "model_name": model_name,

        "accuracy": float(accuracy),

        "precision": float(precision),

        "recall": float(recall),

        "f1_score": float(f1),

        "inference_time_seconds": float(
            inference_time
        ),

        "num_test_samples": int(
            len(y_test)
        ),

        "classification_report": report,

        "confusion_matrix": cm.tolist()
    }

    # ------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------

    print("\n========== MLP METRICS ==========\n")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print(
        f"\nInference Time: "
        f"{inference_time:.4f} seconds"
    )

    print(
        "\n========== CLASSIFICATION REPORT ==========\n"
    )

    print(

        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_
        )
    )

    # SAVE MODEL
    model.save(
        results_dir / f"{model_name}.keras"
    )

    # SAVE METRICS JSON
    metrics_path = (
        results_dir /
        f"{model_name}_metrics.json"
    )

    with open( metrics_path, "w") as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    cm_df = pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    )

    cm_df.to_csv(
        results_dir /
        f"{model_name}_confusion_matrix.csv"
    )

    joblib.dump(
        label_encoder,
        results_dir /
        "label_encoder.joblib"
    )

    print(
        f"\nResults saved in: {results_dir}"
    )

    return metrics

In [9]:
model, history = build_and_train_mlp(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    num_classes=3,
    epochs=50,
    batch_size=16,
    learning_rate=1e-3
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,163 (16.26 KB)

 Trainable params: 4,163 (16.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4109 - loss: 1.0947 - val_accuracy: 0.3067 - val_loss: 1.0697
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4713 - loss: 1.0435 - val_accuracy: 0.4533 - val_loss: 1.0181
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5747 - loss: 0.9630 - val_accuracy: 0.5867 - val_loss: 0.9646
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6149 - loss: 0.9091 - val_accuracy: 0.6267 - val_loss: 0.8932
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6925 - loss: 0.8096 - val_accuracy: 0.6533 - val_loss: 0.8219
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7069 - loss: 0.7642 - val_accuracy: 0.6400 - val_loss: 0.7486
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7356 - loss: 0.6711 - val_accuracy: 0.6933 - val_loss: 0.6829
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7500 - loss: 0.6154 - val_accuracy: 0.7333 - val_loss

In [10]:
metrics = evaluate_model(
    model=model,
    X_test=X_test,
    y_test=y_test,
    label_encoder=label_encoder,
    model_name="kitchen_context_mlp",
    results_dir="results"
)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step

========== MLP METRICS ==========

Accuracy  : 0.9733
Precision : 0.9738
Recall    : 0.9733
F1 Score  : 0.9733

Inference Time: 0.1140 seconds

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

        alto       0.96      1.00      0.98        25
        bajo       1.00      0.96      0.98        25
       medio       0.96      0.96      0.96        25

    accuracy                           0.97        75
   macro avg       0.97      0.97      0.97        75
weighted avg       0.97      0.97      0.97        75


Results saved in: results
